In [0]:
%pip install loguru
%pip install xgboost

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 04 — Model Training & MLflow: XGBoost RUL Prediction

Trains an XGBoost regressor on the Gold feature table to predict
Remaining Useful Life (RUL). Uses unit-aware train/validation splitting
to prevent temporal data leakage.

| Property | Value |
|----------|-------|
| **Algorithm** | XGBoost Regressor |
| **Target** | RUL (capped at 125 cycles) |
| **Split** | 80/20 by engine unit (not random) |
| **Tracking** | MLflow autolog + custom metrics |
| **Registry** | `rul_predictor` → Staging |
| **Metrics** | RMSE, MAE, R², per-bucket accuracy |

**Why split by unit?** Random splitting would leak future sensor readings
from the same engine into training, inflating metrics. In production,
we never have future data from the same haul truck — unit-based splitting
simulates this constraint.

## 1. Configuration

In [0]:
dbutils.widgets.text("gold_path", "/Volumes/workspace/default/gold/features_rul", "Gold Delta path")
dbutils.widgets.text("experiment_name", "/Users/catherine.varas.padilla@gmail.com/rul_experiment", "MLflow experiment")
dbutils.widgets.text("model_name", "workspace.default.rul_predictor", "Model registry name")
dbutils.widgets.text("train_split", "0.8", "Train split ratio (by units)")
dbutils.widgets.text("n_estimators", "200", "XGBoost n_estimators")
dbutils.widgets.text("max_depth", "6", "XGBoost max_depth")
dbutils.widgets.text("learning_rate", "0.1", "XGBoost learning_rate")

GOLD_PATH = dbutils.widgets.get("gold_path")
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")
MODEL_NAME = dbutils.widgets.get("model_name")
TRAIN_SPLIT = float(dbutils.widgets.get("train_split"))
N_ESTIMATORS = int(dbutils.widgets.get("n_estimators"))
MAX_DEPTH = int(dbutils.widgets.get("max_depth"))
LEARNING_RATE = float(dbutils.widgets.get("learning_rate"))

print(f"Gold source:      {GOLD_PATH}")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Model name:        {MODEL_NAME}")
print(f"Train/Val split:   {TRAIN_SPLIT:.0%} / {1 - TRAIN_SPLIT:.0%} by unit")

Gold source:      /Volumes/workspace/default/gold/features_rul
MLflow experiment: /Users/catherine.varas.padilla@gmail.com/rul_experiment
Model name:        workspace.default.rul_predictor
Train/Val split:   80% / 20% by unit


## 2. Imports

In [0]:
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd
import xgboost as xgb
from pyspark.sql import SparkSession
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

spark = SparkSession.builder.getOrCreate()

# Tell MLflow to use Unity Catalog model registry (required in Free Edition with UC)
mlflow.set_registry_uri("databricks-uc")

mlflow.set_experiment(EXPERIMENT_NAME)

2026/04/19 08:46:15 INFO mlflow.tracking.fluent: Experiment with name '/Users/catherine.varas.padilla@gmail.com/rul_experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/3590238286003435', creation_time=1776588375175, experiment_id='3590238286003435', last_update_time=1776588375175, lifecycle_stage='active', name='/Users/catherine.varas.padilla@gmail.com/rul_experiment', tags={'mlflow.experiment.sourceName': '/Users/catherine.varas.padilla@gmail.com/rul_experiment',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'catherine.varas.padilla@gmail.com',
 'mlflow.ownerId': '73183478916021'}>

## 3. Load Gold Feature Table

In [0]:
gold_df = spark.read.format("delta").load(GOLD_PATH)
gold_count = gold_df.count()
print(f"Gold records: {gold_count:,}")
print(f"Columns: {len(gold_df.columns)}")

if gold_count == 0:
    dbutils.notebook.exit("ERROR: Gold table is empty — run notebook 03 first.")

pdf = gold_df.toPandas()
print(f"Units: {pdf['unit_id'].nunique()}")
print(f"RUL range: [{pdf['rul'].min()}, {pdf['rul'].max()}]")

Gold records: 817
Columns: 433
Units: 5
RUL range: [0, 125]


## 4. Train/Validation Split by Unit

**Critical design decision:** We split by equipment unit, not by random row.

A random split would place readings from the same engine in both train and
validation sets. Since consecutive readings from one engine are highly
correlated, this creates **temporal data leakage** — the model effectively
"sees the future" of engines it's being evaluated on.

Unit-based splitting ensures the model has never seen any data from
validation engines, matching the production deployment scenario where
we predict RUL for equipment the model wasn't trained on.

In [0]:
unique_units = pdf["unit_id"].unique()
np.random.seed(42)
np.random.shuffle(unique_units)

split_idx = int(len(unique_units) * TRAIN_SPLIT)
train_units = unique_units[:split_idx]
val_units = unique_units[split_idx:]

train_pdf = pdf[pdf["unit_id"].isin(train_units)].copy()
val_pdf = pdf[pdf["unit_id"].isin(val_units)].copy()

print(f"Train: {len(train_pdf):,} records from {len(train_units)} units")
print(f"Val:   {len(val_pdf):,} records from {len(val_units)} units")
print(f"Train units: {sorted(train_units)[:10]}...")
print(f"Val units:   {sorted(val_units)}")

Train: 648 records from 4 units
Val:   169 records from 1 units
Train units: [1, 2, 3, 5]...
Val units:   [4]


## 5. Prepare Features

In [0]:
EXCLUDE_COLS = {
    "unit_id", "cycle", "rul",
    "_ingestion_timestamp", "_source_file", "_pipeline_version",
    "_silver_timestamp", "_gold_timestamp", "_ingestion_date",
    "timestamp",
}

feature_cols = sorted([c for c in pdf.columns if c not in EXCLUDE_COLS and not c.startswith("_")])

X_train = train_pdf[feature_cols].values.astype(np.float32)
y_train = train_pdf["rul"].values.astype(np.float32)
X_val = val_pdf[feature_cols].values.astype(np.float32)
y_val = val_pdf["rul"].values.astype(np.float32)

print(f"Feature count: {len(feature_cols)}")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")

Feature count: 423
X_train shape: (648, 423)
X_val shape:   (169, 423)


## 6. Train XGBoost with MLflow Tracking

In [0]:
mlflow.xgboost.autolog(log_models=True)

PARAMS = {
    "n_estimators": N_ESTIMATORS,
    "max_depth": MAX_DEPTH,
    "learning_rate": LEARNING_RATE,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "objective": "reg:squarederror",
}

with mlflow.start_run(run_name="xgboost_rul_predictor") as run:
    # Log dataset & split metadata
    mlflow.log_params({
        "dataset": "NASA_CMAPSS_FD001",
        "split_strategy": "by_unit",
        "train_units": len(train_units),
        "val_units": len(val_units),
        "train_records": len(train_pdf),
        "val_records": len(val_pdf),
        "feature_count": len(feature_cols),
        "max_rul_cap": int(pdf["rul"].max()),
    })

    model = xgb.XGBRegressor(**PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False,
    )

    y_pred = model.predict(X_val)

    # Core regression metrics
    rmse = float(np.sqrt(mean_squared_error(y_val, y_pred)))
    mae = float(mean_absolute_error(y_val, y_pred))
    r2 = float(r2_score(y_val, y_pred))

    # Per-bucket accuracy: what % of predictions fall in the correct RUL bucket?
    def bucket_label(rul):
        if rul < 50:
            return "critical_0_50"
        elif rul < 100:
            return "warning_50_100"
        return "healthy_100_plus"

    actual_buckets = np.array([bucket_label(r) for r in y_val])
    pred_buckets = np.array([bucket_label(r) for r in y_pred])

    bucket_accuracy_overall = float(np.mean(actual_buckets == pred_buckets))
    bucket_names = ["critical_0_50", "warning_50_100", "healthy_100_plus"]
    bucket_metrics = {}

    for bucket in bucket_names:
        mask = actual_buckets == bucket
        if mask.sum() > 0:
            acc = float(np.mean(actual_buckets[mask] == pred_buckets[mask]))
            bucket_metrics[f"bucket_accuracy_{bucket}"] = acc

    mlflow.log_metrics({
        "val_rmse": rmse,
        "val_mae": mae,
        "val_r2": r2,
        "bucket_accuracy_overall": bucket_accuracy_overall,
        **bucket_metrics,
    })

    print(f"Validation RMSE:    {rmse:.2f}")
    print(f"Validation MAE:     {mae:.2f}")
    print(f"Validation R²:      {r2:.4f}")
    print(f"Bucket accuracy:    {bucket_accuracy_overall:.4f}")
    for name, acc in bucket_metrics.items():
        print(f"  {name}: {acc:.4f}")

    run_id = run.info.run_id
    print(f"\nMLflow Run ID: {run_id}")

2026/04/19 08:46:33 WARNING mlflow.utils.autologging_utils: You are using an unsupported version of xgboost. If you encounter errors during autologging, try upgrading / downgrading xgboost to a supported version, or try upgrading MLflow.
2026/04/19 08:46:40 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/local_disk0/.ephemeral_nfs/envs/pythonEnv-ec33457f-cca1-4ff6-a59b-10aabf103a36/lib/python3.11/site-packages/xgboost/sklearn.py:1116: UserWarning: [08:46:40] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats."
2026/04/19 08:46:42 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.11/site-packages/_distutils_hack/__init__.py:31: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this conditi

Validation RMSE:    23.02
Validation MAE:     18.04
Validation R²:      0.6930
Bucket accuracy:    0.5207
  bucket_accuracy_critical_0_50: 1.0000
  bucket_accuracy_warning_50_100: 0.7872
  bucket_accuracy_healthy_100_plus: 0.0658

MLflow Run ID: dbb1644b8a994939bdd61cedf76d08eb


## 7. Feature Importance

In [0]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("Top 20 most important features:")
display(spark.createDataFrame(importance_df.head(20)))

Top 20 most important features:


feature,importance
s20_rolling_mean_20,0.411746
s14_rolling_min_20,0.14386015
s14_rolling_mean_20,0.10105726
s21_rolling_max_20,0.05688335
s21_rolling_min_20,0.04699482
s4_rolling_mean_10,0.031536378
s4_rolling_max_10,0.029314436
s11_rolling_mean_20,0.026572628
s4_rolling_mean_20,0.018462712
s4_rolling_max_20,0.01575568


## 8. Register Model to MLflow Registry

Registering with stage `Staging` signals the model is ready for
validation before production deployment.

In [0]:
# Model Registry registration skipped — Free Edition has restricted S3 permissions
# In production (paid Databricks), this would register to Unity Catalog Model Registry:
#   registered_model = mlflow.register_model(f"runs:/{run_id}/model", MODEL_NAME)
#
# For this portfolio MVP, the model is fully tracked via MLflow experiment:
#   - All metrics logged (RMSE, MAE, R², bucket accuracy)
#   - Autologged XGBoost model artifact in the run
#   - Feature importance computed
#   - Reproducible via run_id

model_uri = f"runs:/{run_id}/model"
print(f"Model artifact accessible at: {model_uri}")
print(f"MLflow Run ID: {run_id}")
print("Model Registry registration deferred (Free Edition limitation) — run_id serves as model reference.")

Model artifact accessible at: runs:/dbb1644b8a994939bdd61cedf76d08eb/model
MLflow Run ID: dbb1644b8a994939bdd61cedf76d08eb
Model Registry registration deferred (Free Edition limitation) — run_id serves as model reference.


## 9. Prediction Analysis

In [0]:
pred_analysis = pd.DataFrame({
    "unit_id": val_pdf["unit_id"].values,
    "cycle": val_pdf["cycle"].values,
    "actual_rul": y_val,
    "predicted_rul": y_pred,
})
pred_analysis["residual"] = pred_analysis["actual_rul"] - pred_analysis["predicted_rul"]
pred_analysis["abs_error"] = pred_analysis["residual"].abs()
pred_analysis["actual_bucket"] = pred_analysis["actual_rul"].apply(bucket_label)
pred_analysis["predicted_bucket"] = pred_analysis["predicted_rul"].apply(bucket_label)

print("Prediction error distribution:")
display(spark.createDataFrame(pred_analysis.describe().reset_index()))

Prediction error distribution:


index,unit_id,cycle,actual_rul,predicted_rul,residual,abs_error
count,169.0,169.0,169.0,169.0,169.0,169.0
mean,4.0,99.24260355029585,81.89349365234375,66.86577606201172,15.027709007263184,18.037704467773438
std,0.0,51.33926913275977,41.67119598388672,27.697166442871094,17.488975524902344,14.343819618225098
min,4.0,12.0,0.0,2.8145358562469482,-12.54149055480957,0.15144062042236328
25%,4.0,56.0,46.0,43.74147033691406,-1.0053043365478516,4.7233123779296875
50%,4.0,98.0,91.0,81.19774627685547,14.358692169189453,14.358692169189453
75%,4.0,143.0,125.0,86.58198547363281,31.413673400878906,31.413673400878906
max,4.0,189.0,125.0,108.29570770263672,43.80225372314453,43.80225372314453


## 10. Load Model from Registry & Predict on Test Set

Demonstrates the production inference pattern: load the registered model
by name and stage, then predict on unseen data.

In [0]:
# Load model directly from the run (Free Edition has restricted Model Registry)
loaded_model = mlflow.xgboost.load_model(model_uri)
print(f"Loaded model from run: {run_id}")

test_predictions = loaded_model.predict(X_val)

example_df = pd.DataFrame({
    "unit_id": val_pdf["unit_id"].values[:20],
    "cycle": val_pdf["cycle"].values[:20],
    "actual_rul": y_val[:20],
    "predicted_rul": test_predictions[:20],
    "error": np.abs(y_val[:20] - test_predictions[:20]),
}).round(2)

print("\nExample predictions (first 20 rows):")
display(spark.createDataFrame(example_df))

/local_disk0/.ephemeral_nfs/envs/pythonEnv-ec33457f-cca1-4ff6-a59b-10aabf103a36/lib/python3.11/site-packages/xgboost/sklearn.py:1125: UserWarning: [08:48:30] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)


Loaded model from run: dbb1644b8a994939bdd61cedf76d08eb

Example predictions (first 20 rows):


unit_id,cycle,actual_rul,predicted_rul,error
4,12,125.0,100.7,24.3
4,13,125.0,108.3,16.7
4,14,125.0,100.97,24.03
4,15,125.0,101.75,23.25
4,16,125.0,102.86,22.14
4,17,125.0,97.94,27.06
4,18,125.0,97.2,27.8
4,19,125.0,95.28,29.72
4,20,125.0,94.74,30.26
4,21,125.0,94.55,30.45


## 11. Risk Classification Summary

In [0]:
latest_per_unit = (
    pred_analysis
    .sort_values("cycle")
    .groupby("unit_id")
    .last()
    .reset_index()
)

latest_per_unit["risk_level"] = pd.cut(
    latest_per_unit["predicted_rul"],
    bins=[-1, 50, 100, float("inf")],
    labels=["CRITICAL", "WARNING", "HEALTHY"],
)

risk_dist = latest_per_unit["risk_level"].value_counts()

print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"  Algorithm:        XGBoost ({N_ESTIMATORS} trees, depth {MAX_DEPTH})")
print(f"  Features:         {len(feature_cols)}")
print(f"  Train/Val units:  {len(train_units)} / {len(val_units)}")
print(f"  RMSE:             {rmse:.2f}")
print(f"  MAE:              {mae:.2f}")
print(f"  R²:               {r2:.4f}")
print(f"  Bucket accuracy:  {bucket_accuracy_overall:.4f}")
print(f"  MLflow Run ID:    {run_id}")
print(f"  Risk distribution:")
for level in ["CRITICAL", "WARNING", "HEALTHY"]:
    count = risk_dist.get(level, 0)
    print(f"    {level}: {count} units")
print("=" * 60)

MODEL TRAINING SUMMARY
  Algorithm:        XGBoost (200 trees, depth 6)
  Features:         423
  Train/Val units:  4 / 1
  RMSE:             23.02
  MAE:              18.04
  R²:               0.6930
  Bucket accuracy:  0.5207
  MLflow Run ID:    dbb1644b8a994939bdd61cedf76d08eb
  Risk distribution:
    CRITICAL: 1 units
    HEALTHY: 0 units


In [0]:
display(spark.createDataFrame(latest_per_unit[["unit_id", "predicted_rul", "actual_rul", "abs_error", "risk_level"]].sort_values("predicted_rul")))

unit_id,predicted_rul,actual_rul,abs_error,risk_level
4,3.1578383,0.0,3.1578383,CRITICAL
